# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: OPTIONAL to execute C++ code</h2>
            <span style="color:#f71;">As an alternative, you can run it on the website given yesterday</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use free open source models on Ollama. I also use paid open-source models via Groq and OpenRouter. Only pick the models you want to!
            </span>
        </td>
    </tr>
</table>

In [ ]:
# import libraries

import os
import sys
import io
from dotenv import load_dotenv
from App.config import azure_endpoint, api_version, headers
from openai import OpenAI, AzureOpenAI
import gradio as gr 
import subprocess
from IPython.display import display, update_display, Markdown


In [ ]:
# loading keys

load_dotenv(override=True)

api_key = os.getenv('cd_api_key')
gemini_api_key = os.getenv('GOOGLE_API_KEY')
ollama_api_key = os.getenv('OLLAMA_API_KEY')

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "https://ollama.com/v1"

if api_key and api_key.startswith('e4'):
    print('OpenAI Api key found and looks good so far!')
else:
    print('OpenAI Api key issue, please check')

if gemini_api_key and gemini_api_key.startswith('AIz'):
    print('Gemini Api key found and looks good so far!')
else:
    print('Gemini Api key issue, please check')

if ollama_api_key:
    print('Ollama Api key found and looks good so far!')
else:
    print('Ollama Api key issue, please check')


In [ ]:
openai = AzureOpenAI(api_key=api_key, azure_endpoint=azure_endpoint, api_version=api_version)
gemini = OpenAI(api_key=gemini_api_key, base_url=gemini_url)
ollama = OpenAI(api_key=ollama_api_key, base_url=ollama_url)

gpt_5_mini = 'gpt-5-mini'
gemini_2 = 'gemini-3.1-flash-lite'
ollama_gpt_120b = 'gpt-oss:120b-cloud'
ollama_gpt_20b = 'gpt-oss:20b-cloud'

In [ ]:
def call_model(client, model, messages, **kwargs):
    response = client.chat.completions.create(
        model=model,
        messages = messages,
        **kwargs
    )

    if (kwargs.get('stream')):
        streaming = response

        result = ''
        handle_display = display(Markdown(''), display_id=True)

        for chunk in streaming:

            if not chunk.choices:
                continue
            delta = chunk.choices[0].delta
            result += delta.content or ''
            update_display(Markdown(result), display_id=handle_display.display_id)
    else:
        return response.choices[0].message.content


In [ ]:
models = ['gpt-5', 'gpt-oss:20b-cloud', 'gpt-oss:120b-cloud', 'gemini-3.1-flash-lite']

clients = {'gpt-5': openai, 'gpt-oss:20b-cloud': ollama, 'gpt-oss:120b-cloud': ollama, 'gemini-3.1-flash-lite': gemini}


In [ ]:
from week4.system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

## Overwrite this with the commands from yesterday

Or just use the website like yesterday:

 https://www.programiz.com/cpp-programming/online-compiler/

In [ ]:
compile_command = ["g++", "-O3", "-march=native", "-flto", "-DNDEBUG", "-std=c++20", "week4/main_day4.cpp", "-o", "week4/main_day4.exe"]
run_command =  ["week4/main_day4.exe"]

## And now, on with the main task

In [ ]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called week4/main_day4.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""


In [ ]:
def messages_for(python):
    return [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt_for(python)},
    ]

In [ ]:
def write_output(cpp):
    with open('week4/main_day4.cpp', 'w', encoding='utf-8') as fs:
        fs.write(cpp)

In [ ]:
def port(model, python):
    client = clients[model]

    kwargs = {
        'model': model,
        'messages': messages_for(python)
    }

    if 'gpt' in model: kwargs['reasoning_effort'] = 'high'
    if client is openai: kwargs['extra_headers'] = headers
    response = client.chat.completions.create(**kwargs)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp', '').replace('```', '')
    write_output(reply)
    return reply


In [ ]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [ ]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output


In [ ]:
run_python(pi)

In [ ]:
def compile_and_run():
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    except subprocess.CalledProcessError as e:
        print(f"An error occurred:\n{e.stderr}")

In [ ]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label='Python code:', lines=30, value=pi)
        cpp = gr.Textbox(label='C++ code:', lines=30)
    with gr.Row():
        model = gr.Dropdown(models, label='Select model', value=models[0])
        convert = gr.Button('Convert code')

    convert.click(port, inputs=[model, python], outputs=[cpp])

ui.launch(inbrowser=True)

In [ ]:
compile_and_run()

GPT-5:  0.347552            
GPT-OSS-20B:  0.355741      
GPT-OSS-120B:  0.358380   
GEMINI-3.1-Flash-Lite:   0.361732   